# 🤝 Geração de Dados Sintéticos para Filtragem Colaborativa (v3)

**Disciplina:** Tópicos em Sistemas de Recomendação (UNITINS)
**Autor:** Matheus N.
**Foco:** Construção da matriz de interações usuário-vaga com modelo de afinidade verdadeira (ground truth), exposição desacoplada da opinião.

> Cada decisão de design está documentada em [DECISOES.md](DECISOES.md). O script executável é `executar_simulacao.py`.


## 1. Configuração e Parâmetros

Os parâmetros abaixo foram calibrados após análise da versão original (v2), que produzia ratings dominados por ruído e impedia qualquer modelo de aprender. O design v3 **(Decisão A3)** separa **exposição** (o que o usuário vê) de **opinião** (a nota que dá).

| Parâmetro | Valor | Significado |
|---|---|---|
| N_CATALOGO | 6000 | Vagas no catálogo (amostra estratificada) |
| N_USERS | 5000 | Usuários sintéticos |
| PESO_AFINIDADE_EXPOSICAO | 0,15 | Peso da afinidade na exposição (85% popularidade) |
| TEMPERATURA | 0,25 | Temperatura do softmax (mais alta = mais variedade) |
| RUIDO_RATING | 0,55 | Ruído gaussiano na opinião |
| THRESHOLD_RELEVANCIA | 0,60 | Limiar para considerar uma vaga relevante |


In [1]:
import os, numpy as np, pandas as pd, joblib

SEED = 42; np.random.seed(SEED)

N_CATALOGO = 6000; MIN_VAGAS_POR_PERSONA = 700
N_USERS = 5000
PERSONAS = ['P1_Remoto','P2_Tech','P3_Senior','P4_Junior']
PROB_PERSONAS = [0.25, 0.35, 0.20, 0.20]
PESO_PRIMARY_MIN = 0.55; PESO_PRIMARY_MAX = 0.70
N_INTERACOES_MIN = 80; N_INTERACOES_MAX = 160
PESO_AFINIDADE_EXPOSICAO = 0.15
TEMPERATURA = 0.25; RUIDO_RATING = 0.55; THRESHOLD_RELEVANCIA = 0.60


## 2. Funções Auxiliares

- **`computar_flags_fit`**: define os 4 atributos binários de cada vaga (remoto, tech, sênior, júnior) — mesmos critérios da Etapa 2 (CBF).
- **`computar_popularidade`**: normaliza o número de candidaturas (applies) com log1p, criando $G_j \in [0,1]$.
- **`gerar_vetor_preferencia`**: gera $w_u$ com persona principal entre 0,55–0,70 e as demais dividindo o restante via Dirichlet **(Decisão A5)**.
- **`softmax_sampling`**: amostra itens sem reposição ponderados por softmax(scores/temperatura).


In [2]:
def computar_flags_fit(df):
    titulo = df['title'].fillna('').str.lower()
    nivel = df['formatted_experience_level'].fillna('').astype(str)
    df['fit_p1'] = (df['remote_allowed'].fillna(0).astype(int) == 1).astype(int)
    termos_tech = ['data','scientist','engineer','developer','software','analyst','machine learning','analytics','ai']
    df['fit_p2'] = titulo.apply(lambda t: any(x in t for x in termos_tech)).astype(int)
    df['fit_p3'] = (nivel.isin(['Mid-Senior level','Director','Executive']) | titulo.str.contains('manager|director|lead|head|supervisor|chief',regex=True)).astype(int)
    df['fit_p4'] = (nivel.isin(['Entry level','Internship']) | titulo.str.contains('junior|intern|assistant|trainee|entry',regex=True)).astype(int)
    return df

def computar_popularidade(df):
    applies = df['applies'].fillna(0).clip(lower=0)
    log_app = np.log1p(applies); mx = log_app.max()
    return (log_app / mx).values if mx > 0 else np.zeros(len(df))

def gerar_vetor_preferencia(primary_idx):
    w = np.zeros(4); w[primary_idx] = np.random.uniform(PESO_PRIMARY_MIN, PESO_PRIMARY_MAX)
    outros = np.random.dirichlet(np.ones(3)) * (1 - w[primary_idx])
    for i, val in zip([j for j in range(4) if j != primary_idx], outros): w[i] = val
    return w

def softmax_sampling(scores, temperature, n, replace=True):
    s = np.asarray(scores, dtype=float) - np.max(scores)
    p = np.exp(s / temperature); p = np.clip(p / p.sum(), 1e-15, 1.0); p /= p.sum()
    return np.random.choice(len(scores), size=min(n,len(scores)), replace=replace, p=p)


## 3. Carregar Catálogo e Selecionar Vagas **(Decisão A1)**

Carregamos as 123.849 vagas reais do LinkedIn, computamos flags de persona e popularidade, e selecionamos **6.000 vagas** com amostragem estratificada (garantindo ≥700 por persona). O catálogo menor cria a **sobreposição** que o KNN precisa para encontrar vizinhos.


In [3]:
base_path = "data/raw" if os.path.exists("data/raw") else "."
colunas = ['job_id','title','skills_desc','formatted_experience_level','remote_allowed','applies','views']
df_full = pd.read_csv(f"{base_path}/postings.csv", usecols=colunas).dropna(subset=['title']).reset_index(drop=True)
print(f"Base completa: {len(df_full):,} vagas")
df_full = computar_flags_fit(df_full)
g_j = computar_popularidade(df_full); df_full['popularidade'] = g_j

indices_pp = {p: df_full.index[df_full[f'fit_p{["","1","2","3","4"][i+1]}']==1].tolist() for i,p in enumerate(PERSONAS)}
sel = set()
for p in PERSONAS:
    n = min(MIN_VAGAS_POR_PERSONA, len(indices_pp[p]))
    sel.update(np.random.choice(indices_pp[p], n, replace=False).tolist())
rest = [i for i in df_full.index if i not in sel]
if len(sel) < N_CATALOGO:
    sel.update(np.random.choice(rest, min(N_CATALOGO-len(sel), len(rest)), replace=False).tolist())
df_cat = df_full.loc[sorted(sel)].reset_index(drop=True)
B = df_cat[['fit_p1','fit_p2','fit_p3','fit_p4']].values.astype(float)
G = df_cat['popularidade'].values.astype(float)
N_CAT = len(df_cat)
print(f"Catálogo: {N_CAT} vagas")
for p in PERSONAS:
    col = f'fit_p{["","1","2","3","4"][PERSONAS.index(p)+1]}'
    print(f"  {p}: {int(df_cat[col].sum()):,} ({df_cat[col].mean()*100:.1f}%)")


Base completa: 123,849 vagas
Catálogo: 6000 vagas
  P1_Remoto: 1,356 (22.6%)
  P2_Tech: 1,975 (32.9%)
  P3_Senior: 2,911 (48.5%)
  P4_Junior: 2,196 (36.6%)


## 4. Gerar Usuários e Vetores de Preferência **(Decisão A5)**

Cada usuário recebe uma persona primária (probabilidades 25% Remoto, 35% Tech, 20% Sênior, 20% Júnior) e um vetor $w_u$ contínuo. A persona principal tem peso 0,55–0,70; as demais dividem o restante via Dirichlet. Isso cria **sobreposição gradual** entre personas e dá dimensionalidade ao SVD.


In [4]:
user_primary_idxs = np.random.choice(len(PERSONAS), size=N_USERS, p=PROB_PERSONAS)
user_personas = [PERSONAS[i] for i in user_primary_idxs]
W = np.array([gerar_vetor_preferencia(pi) for pi in user_primary_idxs])
print("Distribuição das personas primárias:")
for i,p in enumerate(PERSONAS):
    print(f"  {p}: {(user_primary_idxs==i).sum()} usuários")


Distribuição das personas primárias:
  P1_Remoto: 1252 usuários
  P2_Tech: 1732 usuários
  P3_Senior: 1017 usuários
  P4_Junior: 999 usuários


## 5. Exposição e Opinião — o coração do design v3 **(Decisão A3)**

**Exposição ≠ Opinião.** O usuário **vê** itens misturando afinidade (15%) e popularidade (85%) via softmax. A **opinião** (rating) reflete **apenas** a afinidade com a persona + ruído.

- `aff_raw = B · w_u` (0 a ~0,70)
- `aff = clip((aff_raw - 0,05) / 0,65, 0, 1)` (normalização para [0,1])
- `exp_score = 0,15·aff + 0,85·G` (exposição mista)
- `rating = clip(round(1 + 4·aff + ε), 1, 5)` com ε ~ N(0, 0,55)

A popularidade na exposição garante que o usuário veja vagas populares que **não** gosta → ratings baixos observados → o modelo aprende também o que o usuário rejeita.


In [5]:
import time
t0 = time.time()
rows = []
for u_id in range(1, N_USERS+1):
    u_idx = u_id - 1
    w_u = W[u_idx]
    aff_raw = np.dot(B, w_u)
    aff = np.clip((aff_raw - 0.05) / 0.65, 0.0, 1.0)
    exp_score = PESO_AFINIDADE_EXPOSICAO * aff + (1 - PESO_AFINIDADE_EXPOSICAO) * G
    n = np.random.randint(N_INTERACOES_MIN, N_INTERACOES_MAX + 1)
    idx = softmax_sampling(exp_score, TEMPERATURA, n, replace=False)
    for v in idx:
        a = aff[v]
        raw = 1.0 + 4.0 * a + np.random.normal(0, RUIDO_RATING)
        r = int(np.clip(round(raw), 1, 5))
        t = 'apply' if r >= 4 else ('view_interested' if r == 3 else 'view_dismissed')
        rows.append({'user_id': u_id, 'job_id': df_cat.iloc[v]['job_id'],
                     'rating': r, 'interaction_type': t, 'user_persona': user_personas[u_idx]})
df = pd.DataFrame(rows)
print(f"Total: {len(df):,} interações em {time.time()-t0:.1f}s")


Total: 602,216 interações em 71.2s


## 6. Diagnóstico da Matriz

Verificamos a qualidade dos dados gerados: esparsidade, distribuição de ratings, cobertura de itens e fração de relevantes por usuário. Uma distribuição equilibrada (notas 1–2 e 4–5 em proporções semelhantes) é sinal de que o desacoplamento exposição-opinião funcionou.


In [6]:
n_u = df['user_id'].nunique(); n_j = df['job_id'].nunique()
espars = (1 - len(df) / (n_u * n_j)) * 100
print(f"Usuários: {n_u:,}  |  Vagas interagidas: {n_j:,}  |  Esparsidade: {espars:.2f}%")
print(f"\nDistribuição de ratings:")
for r, cnt in df['rating'].value_counts().sort_index().items():
    print(f"  {r}: {cnt:,} ({cnt/len(df)*100:.1f}%)")
print(f"  ≥4: {df['rating'].ge(4).mean()*100:.1f}%  |  ≤2: {df['rating'].le(2).mean()*100:.1f}%")
ipu = df.groupby('user_id').size()
print(f"\nInterações/user: média={ipu.mean():.1f}, min={ipu.min()}, max={ipu.max()}")
rpi = df.groupby('job_id').size()
print(f"Ratings/item: média={rpi.mean():.1f}, mediana={rpi.median():.0f}, min={rpi.min()}, max={rpi.max()}")
print(f"  Itens com <5 ratings: {(rpi<5).sum()} ({(rpi<5).mean()*100:.1f}%)")

# Relevantes por usuário (amostra)
n_amostra = 200
rels = []
for u_idx in range(min(n_amostra, N_USERS)):
    aff = np.clip((np.dot(B, W[u_idx]) - 0.05) / 0.65, 0.0, 1.0)
    rels.append((aff >= THRESHOLD_RELEVANCIA).sum())
rels = np.array(rels)
print(f"\nRelevantes (aff≥{THRESHOLD_RELEVANCIA})/user: média={rels.mean():.0f}, "
      f"min={rels.min()}, max={rels.max()}")


Usuários: 5,000  |  Vagas interagidas: 6,000  |  Esparsidade: 97.99%

Distribuição de ratings:
  1: 122,291 (20.3%)
  2: 114,547 (19.0%)
  3: 66,082 (11.0%)
  4: 89,007 (14.8%)
  5: 210,289 (34.9%)
  ≥4: 49.7%  |  ≤2: 39.3%

Interações/user: média=120.4, min=80, max=160
Ratings/item: média=100.4, mediana=72, min=32, max=1228
  Itens com <5 ratings: 0 (0.0%)

Relevantes (aff≥0.6)/user: média=2073, min=1356, max=2984


## 7. Exportar Arquivos

Salvamos a matriz de interações (mesmo formato da versão anterior, para compatibilidade com o dashboard) e o **ground truth** (`verdade_afinidade.pkl`) contendo $W$, $B$, $G$, job_ids e parâmetros — usado pelo script de modelagem para avaliar Precision@K e NDCG@K contra a verdade real.


In [7]:
df.to_csv("data/processed/interacoes_sinteticas.csv", index=False)
joblib.dump({
    'W': W, 'B': B, 'G': G,
    'job_ids': df_cat['job_id'].values,
    'user_personas': user_personas,
    'params': {'lambda_pop': 0.15, 'peso_primary_min': PESO_PRIMARY_MIN,
               'peso_primary_max': PESO_PRIMARY_MAX, 'n_catalogo': N_CAT,
               'n_users': N_USERS, 'threshold_relevancia': THRESHOLD_RELEVANCIA,
               'ruido_rating': RUIDO_RATING, 'temperatura': TEMPERATURA,
               'peso_afinidade_exposicao': PESO_AFINIDADE_EXPOSICAO,
               'n_interacoes_min': N_INTERACOES_MIN, 'n_interacoes_max': N_INTERACOES_MAX,
               'seed': SEED}
}, "data/processed/verdade_afinidade.pkl")
print("\n✅ interacoes_sinteticas.csv + verdade_afinidade.pkl exportados!")



✅ interacoes_sinteticas.csv + verdade_afinidade.pkl exportados!
